# Tutorial: Clasificación de abandono de empleado (Employee Churn Model)

## Contexto del ejercicio
En este notebook construiremos un modelo de clasificación para estimar la probabilidad de que un empleado abandone la organización. Este tipo de análisis es útil en iniciativas de *People Analytics*, retención de talento y analítica predictiva aplicada a recursos humanos.

## Audiencia
- Alumnos de posgrado y educación continua que desean practicar clasificación supervisada con datos tabulares.

## Prerrequisitos
- Conocimientos básicos de Python y pandas.
- Nociones generales de entrenamiento y evaluación de modelos de clasificación.

## Objetivo
Construir un modelo base de **abandono de empleado** con `scikit-learn`, realizar un EDA previo y discutir la interpretación de los resultados desde una perspectiva técnica y de negocio.


## Ruta del ejercicio
1. Descargar el archivo CSV desde el portafolio.
2. Subir el dataset manualmente a Google Colab.
3. Revisar la estructura del conjunto de datos.
4. Realizar un análisis exploratorio de datos (EDA).
5. Preparar variables numéricas y categóricas.
6. Entrenar un modelo base con `LogisticRegression`.
7. Evaluar el modelo e interpretar los hallazgos.


## Uso recomendado en Google Colab
Para este ejercicio se recomienda que el alumno **descargue el archivo CSV** desde el portafolio y luego lo **suba manualmente a Google Colab**.

Esto ayuda a que el alumno se familiarice con un flujo de trabajo común en Colab cuando se trabaja con archivos externos.

**Nombre esperado del archivo:** `HR_dataset_copy.csv`


In [ ]:
import sys

if 'google.colab' in sys.modules:
    try:
        import sklearn  # noqa: F401
    except ImportError:
        %pip install -q scikit-learn pandas matplotlib seaborn

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('deep')


In [ ]:
LOCAL_CANDIDATES = [
    Path('../data/employee-churn/HR_dataset_copy.csv'),
    Path('data/employee-churn/HR_dataset_copy.csv'),
    Path('HR_dataset_copy.csv'),
    Path('/content/HR_dataset_copy.csv')
]

def load_dataset() -> pd.DataFrame:
    if 'google.colab' in sys.modules:
        from google.colab import files
        print('Seleccione el archivo HR_dataset_copy.csv desde su computadora.')
        uploaded = files.upload()
        uploaded_name = next(iter(uploaded))
        return pd.read_csv(uploaded_name)

    for candidate in LOCAL_CANDIDATES:
        if candidate.exists():
            return pd.read_csv(candidate)

    raise FileNotFoundError('No se encontró HR_dataset_copy.csv en las rutas locales esperadas.')

df = load_dataset()
df.head()


## Paso 1. Revisión inicial del dataset
Antes de modelar, conviene entender cuántos registros tenemos, qué columnas existen, qué tipos de datos aparecen y si hay valores faltantes.


In [ ]:
print(f'Registros: {df.shape[0]:,}')
print(f'Columnas: {df.shape[1]}')
display(df.dtypes.to_frame('tipo_de_dato'))
display(df.isna().sum().to_frame('valores_faltantes'))


## Paso 2. Definición de la variable objetivo
La variable `left` indica si el empleado salió de la organización (`1`) o permaneció (`0`). Esta será la variable objetivo del modelo.


In [ ]:
target = 'left'
target_distribution = df[target].value_counts().sort_index()
target_share = df[target].value_counts(normalize=True).sort_index()
display(pd.DataFrame({'conteo': target_distribution, 'proporción': target_share}))


In [ ]:
ax = target_distribution.plot(kind='bar', color=['#4c78a8', '#f58518'], figsize=(6, 4))
ax.set_title('Distribución de la variable objetivo: left')
ax.set_xlabel('Clase')
ax.set_ylabel('Número de empleados')
plt.show()


### Interpretación inicial
Esta distribución permite evaluar si el problema está desbalanceado. Si una de las clases domina mucho, entonces métricas como *accuracy* pueden ser engañosas y será necesario prestar especial atención a *precision*, *recall* y la matriz de confusión.


## Paso 3. EDA: análisis exploratorio de datos
A continuación revisaremos algunos patrones descriptivos antes de entrenar el modelo. Esto ayuda a formular hipótesis sobre los factores asociados al abandono.


In [ ]:
summary_cols = ['satisfaction_level', 'last_evaluation', 'number_project', 'average_montly_hours', 'time_spend_company']
df[summary_cols].describe().T


In [ ]:
plt.figure(figsize=(7, 4))
sns.histplot(data=df, x='satisfaction_level', hue='left', bins=20, kde=True, element='step')
plt.title('Distribución de satisfacción por abandono')
plt.xlabel('Satisfaction level')
plt.ylabel('Frecuencia')
plt.show()


In [ ]:
plt.figure(figsize=(7, 4))
sns.boxplot(data=df, x='left', y='average_montly_hours')
plt.title('Horas mensuales promedio según abandono')
plt.xlabel('Abandono (left)')
plt.ylabel('Average monthly hours')
plt.show()


In [ ]:
salary_churn = df.groupby('salary')['left'].mean().sort_values(ascending=False)
salary_churn.plot(kind='bar', figsize=(6, 4), color='#f58518')
plt.title('Tasa de abandono por nivel salarial')
plt.xlabel('Salary')
plt.ylabel('Proporción de abandono')
plt.show()


In [ ]:
area_churn = df.groupby('functional area')['left'].mean().sort_values(ascending=False)
area_churn.plot(kind='bar', figsize=(10, 4), color='#54a24b')
plt.title('Tasa de abandono por área funcional')
plt.xlabel('Functional area')
plt.ylabel('Proporción de abandono')
plt.xticks(rotation=45, ha='right')
plt.show()


In [ ]:
corr = df.select_dtypes(include=['number']).corr(numeric_only=True)
plt.figure(figsize=(8, 6))
sns.heatmap(corr, cmap='coolwarm', annot=True, fmt='.2f')
plt.title('Matriz de correlación de variables numéricas')
plt.show()


### Lectura del EDA
Estas visualizaciones permiten identificar patrones preliminares. Por ejemplo, puede observarse si los empleados que abandonan presentan menor satisfacción, más horas trabajadas o una mayor concentración en ciertas áreas funcionales o niveles salariales.

Este tipo de lectura es importante porque ayuda a conectar el modelo con posibles decisiones de negocio o de gestión del talento.


## Paso 4. Preparación del modelo
Usaremos la variable `left` como objetivo. Las columnas numéricas y categóricas se transforman dentro de un `Pipeline` para dejar el flujo reproducible y compatible con Google Colab.


In [ ]:
X = df.drop(columns=[target])
y = df[target]

numeric_features = X.select_dtypes(include=['number']).columns.tolist()
categorical_features = X.select_dtypes(exclude=['number']).columns.tolist()

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median'))
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features),
    ]
)

model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000, random_state=42))
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

print(f'Tamaño entrenamiento: {X_train.shape[0]:,}')
print(f'Tamaño prueba: {X_test.shape[0]:,}')
print(f'Variables numéricas: {len(numeric_features)}')
print(f'Variables categóricas: {len(categorical_features)}')


## Paso 5. Entrenamiento del modelo
Usaremos una regresión logística como modelo base. Es una opción adecuada para empezar porque es interpretable, rápida y muy útil como punto de comparación.


In [ ]:
model.fit(X_train, y_train)
pred = model.predict(X_test)
proba = model.predict_proba(X_test)[:, 1]

accuracy = accuracy_score(y_test, pred)
roc_auc = roc_auc_score(y_test, proba)

print(f'Accuracy: {accuracy:.4f}')
print(f'ROC AUC: {roc_auc:.4f}')
print()
print(classification_report(y_test, pred))


### Cómo interpretar estas métricas
- **Accuracy:** proporción total de predicciones correctas.
- **Precision:** de los empleados marcados como abandono, cuántos realmente abandonaron.
- **Recall:** de los empleados que realmente abandonaron, cuántos fueron detectados por el modelo.
- **ROC AUC:** capacidad general del modelo para distinguir entre permanencia y abandono a distintos umbrales.

En problemas de retención suele ser especialmente importante vigilar el **recall** de la clase de abandono, porque un falso negativo significa no detectar a tiempo a un empleado con riesgo de salida.


In [ ]:
cm = confusion_matrix(y_test, pred)
cm_df = pd.DataFrame(cm, index=['Real 0', 'Real 1'], columns=['Pred 0', 'Pred 1'])
display(cm_df)

plt.figure(figsize=(6, 4))
sns.heatmap(cm_df, annot=True, fmt='d', cmap='Blues')
plt.title('Matriz de confusión')
plt.show()


### Interpretación de la matriz de confusión
- **Pred 1 / Real 1:** empleados con abandono correctamente detectados.
- **Pred 0 / Real 1:** empleados que abandonaron pero el modelo no detectó.
- **Pred 1 / Real 0:** empleados marcados con riesgo aunque en realidad permanecieron.

Desde una perspectiva de negocio, los falsos negativos suelen ser más costosos si la organización quiere intervenir antes de perder talento clave.


In [ ]:
feature_names = model.named_steps['preprocessor'].get_feature_names_out()
coefficients = model.named_steps['classifier'].coef_[0]
coef_df = (
    pd.DataFrame({'feature': feature_names, 'coef': coefficients})
    .assign(abs_coef=lambda d: d['coef'].abs())
    .sort_values('abs_coef', ascending=False)
)
coef_df[['feature', 'coef']].head(12)


In [ ]:
top_positive = coef_df.sort_values('coef', ascending=False).head(5)[['feature', 'coef']]
top_negative = coef_df.sort_values('coef', ascending=True).head(5)[['feature', 'coef']]

print('Variables más asociadas con abandono (coeficientes positivos):')
display(top_positive)

print('Variables más asociadas con permanencia (coeficientes negativos):')
display(top_negative)


## Interpretación de variables relevantes
En una regresión logística, un coeficiente positivo empuja la predicción hacia `left = 1` y un coeficiente negativo hacia `left = 0`.

Esto no debe leerse como causalidad directa, sino como una señal estadística dentro de este dataset. Conviene complementar estos hallazgos con conocimiento del proceso, entrevistas y políticas de recursos humanos.


## Conclusión ejecutiva
Este notebook deja una línea base reproducible para el problema de abandono de empleado. A partir de los resultados obtenidos, el grupo puede discutir temas como:
- La utilidad del modelo para priorizar intervenciones de retención.
- La conveniencia de optimizar el modelo hacia mayor recall o mayor precisión.
- Las variables que merecen revisión por parte del área de recursos humanos.

El siguiente paso natural sería comparar este modelo con alternativas como árboles, bosques aleatorios o *gradient boosting*.


## Ejercicio para el alumno
Pruebe una de estas extensiones:
1. Cambiar `LogisticRegression` por `RandomForestClassifier`.
2. Ajustar el umbral de clasificación usando `predict_proba`.
3. Comparar resultados quitando la columna categórica `functional area`.
4. Evaluar si la satisfacción del empleado parece ser una variable especialmente sensible.


In [ ]:
# Respuesta sugerida: use este espacio para probar un segundo modelo o un nuevo umbral.
# Ejemplo:
# from sklearn.ensemble import RandomForestClassifier
# ...


## Errores comunes y extensiones
**Error común:** olvidar el tratamiento de variables categóricas y pasar texto crudo al modelo.

**Error común:** quedarse solo con accuracy y no revisar recall, precisión o matriz de confusión.

**Extensión sugerida:** agregar validación cruzada, comparar varios clasificadores y documentar cuál conviene presentar como modelo final según el objetivo del negocio.
